1. Импортируем модуль os для работы с операционной системой (лабораторная работа выполнялась под Windows).

In [395]:
import os

2. Задаем путь к директории hadoop под Windows (winutils.exe и hadoop.dll), записываем его в переменную окружения HADOOP_HOME, добавляем ее PATH, чтобы при запуске процесса была найдена hadoop.dll для работы Spark.

In [396]:
_HADOOP_HOME = r"C:\hadoop"
os.environ["HADOOP_HOME"] = _HADOOP_HOME
os.environ["PATH"] = _HADOOP_HOME + r"\bin;" + os.environ.get("PATH", "")

3. Импортируем необходимые модули из pyspark.sql.

In [397]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

4. Создаем сессию Spark для всего JupyterNotebook.

In [398]:
spark = (
    SparkSession.builder
    .appName("6403_KuchumovAA_BigData_Lab_2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.extraLibraryPath", _HADOOP_HOME + r"\bin")
    .getOrCreate()
)

5. Собираем данные о языках программирования в виде Spark DataFrame:
    - опция header=true - первая строка файла интерпретируется как заголовок (name,wikipedia_url)
    - чтение из data/programming-languages.csv
    - столбец name:
        - trim: убираем пробелы в начале и в конце каждой строки
        - lower: переводим строку в нижний регистр
        - alias: превращаем имя столбца из lower(trim(name)) в language_name

In [399]:
languages_df = (
    spark.read
    .option("header", "true")
    .csv("data/programming-languages.csv")
    .select(F.lower(F.trim(F.col("name"))).alias("language_name"))
    .distinct()
)

print(languages_df.show(5))

+-------------+
|language_name|
+-------------+
|      a# .net|
|          abc|
|    abc algol|
|        abset|
|     ace dasl|
+-------------+
only showing top 5 rows

None


6. Читаем файл как текстовый в сырой DataFrame: каждая строка файла - строка таблицы с колонкой value.

In [400]:
raw_df = spark.read.text("data/posts_sample.xml")
print(raw_df.show(5))

+--------------------+
|               value|
+--------------------+
|<?xml version="1....|
|             <posts>|
|  <row Id="4" Pos...|
|  <row Id="6" Pos...|
|  <row Id="7" Pos...|
+--------------------+
only showing top 5 rows

None


7. Из сырого DataFrame оставляем строки, которые удовлетворяют следующим условиям:
    - содержание подстроки "<row" - отсекаем служебные строки (например, <?xml ...>)
    - содержание подстроки "PostTypeId="1"" - в дампе Stack Overflow это вопросы (ответы имеют "PostTypeId="2"")

In [401]:
questions_rows = raw_df.filter(
    F.col("value").contains("<row") &
    F.col("value").contains('PostTypeId="1"')
)
print(questions_rows.show(5))

+--------------------+
|               value|
+--------------------+
|  <row Id="4" Pos...|
|  <row Id="6" Pos...|
|  <row Id="9" Pos...|
|  <row Id="11" Po...|
|  <row Id="13" Po...|
+--------------------+
only showing top 5 rows

None


8. Выбираем строки с помощью select с годом и тегами, извлеченными с помощью регулярных выражений, а с помощью filter оставляем только те строки, в которых значение года находится в диапазоне от 2010 до 2020. 

In [402]:
parsed_tags_rows = questions_rows.select(
    F.regexp_extract("value", r'CreationDate="(\d{4})', 1)
        .cast("int").alias("year"),
    F.regexp_extract("value", r'Tags="([^"]*)"', 1).alias("tags"),
).filter(
    F.col("year").between(2010, 2020) & (F.col("tags") != "")
)
print(parsed_tags_rows.show(5))

+----+--------------------+
|year|                tags|
+----+--------------------+
|2010|&lt;c++&gt;&lt;ch...|
|2010|&lt;sharepoint&gt...|
|2010|&lt;iphone&gt;&lt...|
|2010|&lt;symfony1&gt;&...|
|2010|        &lt;java&gt;|
+----+--------------------+
only showing top 5 rows

None


9. Преобразуем набор тегов в каждой строке в читаемый вид с помощью регулярных выражений.

In [403]:
str_tags_rows = parsed_tags_rows.withColumn(
    "tags_str",
    F.regexp_replace(
        F.regexp_replace("tags", "&lt;", "<"),
        "&gt;", ">"
    )
)
print(str_tags_rows.show(5))

+----+--------------------+--------------------+
|year|                tags|            tags_str|
+----+--------------------+--------------------+
|2010|&lt;c++&gt;&lt;ch...|<c++><character-e...|
|2010|&lt;sharepoint&gt...|<sharepoint><info...|
|2010|&lt;iphone&gt;&lt...|<iphone><app-stor...|
|2010|&lt;symfony1&gt;&...|<symfony1><schema...|
|2010|        &lt;java&gt;|              <java>|
+----+--------------------+--------------------+
only showing top 5 rows

None


10. Строим новую таблицу, где каждый отдельный тег становится отдельной строкой с тем же годом.

In [404]:
clean_tags_rows = str_tags_rows.select(
    "year",
    F.explode(
        F.split(
            F.regexp_replace("tags_str", r"^<|>$", ""),
            r"><"
        )
    ).alias("tag")
).select("year", F.lower("tag").alias("tag"))
print(clean_tags_rows.show(5))

+----+------------------+
|year|               tag|
+----+------------------+
|2010|               c++|
|2010|character-encoding|
|2010|        sharepoint|
|2010|          infopath|
|2010|            iphone|
+----+------------------+
only showing top 5 rows

None


11. С помощью внутреннего соединения оставляем только те строки, теги в которых содержат язык программирования, а с помощью select оставляем только год и название языка программирования.
    - join(...) - таблица с парами год-тег соединяется с таблицей со списком языков программирования по условию тег совпадает с языком программирования; inner: в итоговую таблицу попадают только строки, для которых нашлась пара в списке языков программирования

In [405]:
clean_language_rows = (
    clean_tags_rows
    .join(languages_df, clean_tags_rows["tag"] == languages_df["language_name"], "inner")
    .select("year", F.col("tag").alias("language"))
)
print(clean_language_rows.show(5))

+----+--------+
|year|language|
+----+--------+
|2010|    java|
|2010|     php|
|2010|    ruby|
|2010|       c|
|2010|     php|
+----+--------+
only showing top 5 rows

None


12. С помощью группировки и последующей функцией агрегации данных (подсчет количества) считаем, сколько раз каждый язык программирования встретился в каждом году.
    - groupBy(...) - строки с одинаковой парой значений года и названия языка программирования группируются
    - agg(...) - считаем число строк в каждой группе

In [406]:
language_counts = (
    clean_language_rows
    .groupBy("year", "language")
    .agg(F.count("*").alias("count"))
)
print(language_counts.show(5))

+----+----------+-----+
|year|  language|count|
+----+----------+-----+
|2010|      java|   52|
|2010|       php|   46|
|2010|         c|   20|
|2010|    python|   26|
|2010|javascript|   44|
+----+----------+-----+
only showing top 5 rows

None


13. Задаем "окно": все строки с одним и тем же значением года образуют одну партицию.
    - partitionBy(...) - окно считается отдельно внутри каждого значения года ("разбиение" таблицы на группы по годам)
    - orderBy(...) - внутри одного года строки упорядочиваются по убыванию количества упоминаний языка программирования

In [407]:
year_window = Window.partitionBy("year").orderBy(F.desc("count"))

14. Из агрегата строим итоговую таблицу - топ языков по каждому году:
    - withColumn(...) - добавляет новую колонку rank, значение считается функцией rank() по заданному окну: внутри одного года строки упорядочены по убыванию счётчика, и каждой паре год - язык присваивается место в рейтинге (с учётом ничьих у rank)
    - filter(...) - оставляет только строки, где rank не больше 10
    - select(...) - формирует финальный набор колонок: год, ранг, название языка программирование и количество упоминаний
    - orderBy(...) - cортирует весь DataFrame: сначала по возрастанию year, при равном годе - по возрастанию rank, чтобы в выводе шли год за годом, а внутри года - места 1, 2, 3, ...

In [408]:
result = (
    language_counts
    .withColumn("rank", F.rank().over(year_window))
    .filter(F.col("rank") <= 10)
    .select("year", "rank", "language", "count")
    .orderBy("year", "rank")
)

15. Записываем данные в .parquet-файл и выводим его директорию.

In [409]:
output_path = "output/"
result.write.mode("overwrite").parquet(output_path)
print(f"\nФайл .parquet сохранен по пути - {output_path}\n")


Файл .parquet сохранен по пути - output/



16. Читаем данные из .parquet-файла.

In [410]:
result = spark.read.parquet(output_path).orderBy("year", "rank")

17. Выводим данные, извлеченные из .parquet-файла.

In [411]:
years = sorted([r["year"] for r in result.select("year").distinct().collect()])
for y in years:
    print()
    print(f"  Год {y} - Топ-10 языков программирования")
    print()
    print(f"  {'Место':<6} {'Язык программирования':<22} {'Число постов':>8}")
    print()
    for row in result.filter(F.col("year") == y).orderBy("rank").collect():
        print(f"  {row['rank']:<6} {row['language']:<22} {row['count']:>8}")


  Год 2010 - Топ-10 языков программирования

  Место  Язык программирования  Число постов

  1      java                         52
  2      php                          46
  3      javascript                   44
  4      python                       26
  5      objective-c                  23
  6      c                            20
  7      ruby                         12
  8      delphi                        8
  9      applescript                   3
  9      r                             3
  9      bash                          3
  9      perl                          3

  Год 2011 - Топ-10 языков программирования

  Место  Язык программирования  Число постов

  1      php                         102
  2      java                         93
  3      javascript                   83
  4      python                       37
  5      objective-c                  34
  6      c                            24
  7      ruby                         20
  8      perl                        

18. Завершаем сессию Spark.

In [412]:
spark.stop()